# Sane — read before you answer

Two English language models trained from scratch at [Sane Labs](https://sanelabs.org):
**Sane-47M** and **Sane-118M**. Original architecture, original tokenizer, original
training code — no existing weights were fine-tuned, distilled or merged into them.
Apache-2.0.

They are small on purpose. A model this size cannot store facts, so these are trained to
**read**: you hand one a passage and it answers out of that passage, and tells you when
the answer is not in there.

Run the cell below (▶), wait about a minute for the download, then play with the last cell.
A free CPU runtime is enough — no GPU needed.


In [ ]:
#@title Setup — install, download, load  { display-mode: "form" }
MODEL = "sekund0chka/sane-118m" #@param ["sekund0chka/sane-118m", "sekund0chka/sane-47m"]

!pip -q install torch tokenizers huggingface_hub

import importlib.util, torch
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

# The models ship with their own definition; import it rather than retyping it here.
spec = importlib.util.spec_from_file_location("sane", hf_hub_download(MODEL, "sane_chat.py"))
sane = importlib.util.module_from_spec(spec); spec.loader.exec_module(sane)

sd  = torch.load(hf_hub_download(MODEL, "sane_final_fp16.pt"), map_location="cpu", weights_only=False)
tok = Tokenizer.from_file(hf_hub_download(MODEL, "tokenizer.json"))
cfg = sd["cfg"]

model = sane.GPT(cfg["vocab"], cfg["d_model"], cfg["n_layer"], cfg["n_head"], cfg["ctx"], cfg["hidden"])
model.load_state_dict({k: v.float() for k, v in sd["model"].items()})
model.eval()
END, USER = tok.token_to_id("<|end|>"), tok.token_to_id("<|user|>")
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters, step {sd['step']}, ready")


In [ ]:
#@title The two functions this whole thing is about
@torch.no_grad()
def generate(prompt, max_new=60, temperature=0.0):
    ids = tok.encode(prompt).ids
    x, out, seen = torch.tensor([ids]), [], list(ids)
    for _ in range(max_new):
        logits = model(x[:, -model.ctx:])[0, -1].float()
        idx = torch.tensor(sorted(set(seen[-256:])))       # repetition penalty
        l = logits[idx]
        logits[idx] = torch.where(l > 0, l / 1.12, l * 1.12)
        if temperature <= 0.01:
            nxt = int(torch.argmax(logits))
        else:
            logits = logits / temperature
            v, _ = torch.topk(logits, 50)
            logits[logits < v[-1]] = -float("inf")
            nxt = int(torch.multinomial(torch.softmax(logits, -1), 1))
        if nxt in (END, USER):
            break
        out.append(nxt); seen.append(nxt)
        x = torch.cat([x, torch.tensor([[nxt]])], dim=1)
    return tok.decode(out).strip()

def ask(sources, question, temperature=0.0):
    prompt = (f"<|context|>\n{sources}\n" if sources.strip() else "") + \
             f"<|user|>\n{question}\n<|sane|>\n"
    return generate(prompt, temperature=temperature)


## What it looks like when it works

The sources below contain the weight and the price, plus two numbers designed to pull the
answer off course. The second question asks for something that simply is not in them.


In [ ]:
SOURCES = (
    "Search results: The Ridgeloom V4 weighs 34 kg and sells for 690 euros. "
    "A forum post says a crate came in at about 50 kg. The older V3 weighed 41 kg."
)

for q in ["How much does the Ridgeloom V4 weigh?",
          "What colour is the Ridgeloom V4?"]:
    print("Q:", q)
    print("A:", ask(SOURCES, q), "\n")


## Your turn

Paste anything into `sources` — documentation, an article, a page of search results —
and ask about it. Then ask something it does not cover, and see whether the model admits
it. Leave `sources` empty to watch a 118M-parameter model with nothing to read invent a
fact with total confidence; that is the wall all of this is aimed at.


In [ ]:
#@title Ask it something  { display-mode: "form" }
sources = "Search results: Canberra is the capital city of Australia. The city was laid out in 1913 by Walter Burley Griffin, after an international design competition." #@param {type:"string"}
question = "When was Canberra laid out?" #@param {type:"string"}
temperature = 0 #@param {type:"slider", min:0, max:1.2, step:0.1}

print(ask(sources, question, temperature))


## Where it breaks

You will find these anyway, so they are written down here first.

* **With nothing to read it invents facts**, fluently. Capacity wall, not a tuning problem:
  an extra round of factual fine-tuning produced more pattern-matched leakage, not more
  knowledge.
* **It errs towards refusal too** and will sometimes tell you a fact is missing when the
  sources contain it.
* **It reads plain prose much better than a heavily formatted results page** with
  `[1] Title — subtitle` headers on every entry. That is a limit of its behaviour data,
  not of the idea.
* 1024-token window, English only, no safety tuning of any kind.

The next model, [Synth-2](https://sanelabs.org/synth-2/), is 1.23B parameters with ~330M
active per token, and carries the idea into the architecture instead of the training data:
a small head on every layer predicts the model's own error for each token, so an answer can
underline the span you should check instead of asking to be trusted whole. It is in
pretraining, and its run log and [working notes](https://sanelabs.org/notes/) are public,
including the measurements that killed a favourite explanation.

Model cards: [Sane-118M](https://huggingface.co/sekund0chka/sane-118m) ·
[Sane-47M](https://huggingface.co/sekund0chka/sane-47m). Corrections to any number are
welcome — ssanelabs@gmail.com
